# Módulo 2 Ejercicios de R
### Visualización, Manipulación de Datos y Análisis Estadístico

**Autor:** Camilo Romero

---

Este notebook presenta la solución desarrollada para los ejercicios del módulo en **R**, incorporando limpieza de datos, transformaciones tabulares con `tidyr`/`dplyr`, tablas descriptivas y visualizaciones con `ggplot2` y `plotly`.

**Contenido:**
1. [Ejercicio 1 — Serie temporal de acres quemados por causa en Idaho](#ejercicio-1)
2. [Ejercicio 2 — Análisis de atletas medallistas en Río 2016](#ejercicio-2)
   - [2.1 Deportes con mayor cantidad de medallas](#21-deportes-con-mayor-número-de-medallas)
   - [2.2 Distribución de edad por deporte](#22-distribución-de-la-edad)
   - [2.3 Equipos nacionales destacados](#23-equipos-nacionales-con-más-medallas)
   - [2.4 Análisis y dispersión del peso corporal](#24-análisis-del-peso-por-sexo)
3. [Ejercicio 3 — Reestructuración tabular de población por estado](#ejercicio-3)
4. [Ejercicio 4 — Histograma de distribución de grandes incendios](#ejercicio-4)

**Librerías utilizadas:** `dplyr`, `ggplot2`, `tidyr`, `readr`, `plotly`


In [1]:
# Carga de librerías del ecosistema tidyverse y visualización
library(dplyr)
library(ggplot2)
library(tidyr)
library(readr)
library(plotly)

Attaching package: 'dplyr'

The following objects are masked from 'package:stats':

    filter, lag

The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union



---
## Ejercicio 1 {#ejercicio-1}

Evolución histórica de los acres quemados por incendios forestales en el estado de **Idaho**, clasificados por tipo de causa a lo largo de los años mediante una serie temporal interactiva con `plotly`.

In [2]:
df_fires <- read.csv("StudyArea.csv")

grafico <- df_fires %>%
  filter(STATE == "Idaho") %>%
  mutate(CAUSE = ifelse(is.na(CAUSE) | trimws(CAUSE) == "", 
                        "Sin información", CAUSE)) %>%
  select(YEAR_, CAUSE, TOTALACRES) %>%
  group_by(YEAR_, CAUSE) %>%
  summarise(totalacres = sum(TOTALACRES, na.rm = TRUE), .groups = "drop") %>%
  ggplot(mapping = aes(x = YEAR_, y = totalacres, color = CAUSE)) +
  geom_line(linewidth = 0.8) +
  geom_point(size = 1.5) +
  labs(
    title = "Evolución de acres quemados en Idaho por causa",
    x = "Año",
    y = "Total de acres quemados",
    color = "Causa"
  ) +
  theme_bw()

ggplotly(grafico)

[Plotly / ggplot2 Series Temporales: Total de acres quemados vs Año en Idaho por Causa]

---
## Ejercicio 2 {#ejercicio-2}

Análisis descriptivo del conjunto de datos de atletas participantes en los Juegos Olímpicos de **Río 2016** (`athlete_events.csv`).

### 2.1 Deportes con mayor número de medallas
Identificación del Top 5 de disciplinas deportivas con mayor volumen de medallas entregadas.

In [3]:
df_olympics <- read.csv("athlete_events.csv")

df_2016 <- df_olympics %>%
  filter(Year == 2016, !is.na(Medal))

tabla_medallas <- df_2016 %>%
  count(Sport, sort = TRUE) %>%
  head(5)

tabla_medallas

,Sport,n
1,Athletics,192
2,Swimming,191
3,Rowing,144
4,Football,106
5,Hockey,99


### 2.2 Distribución de la edad
Cálculo de medidas de tendencia central y dispersión para la edad de los medallistas en los cinco deportes principales.

In [4]:
df_top_sports <- df_2016 %>%
  filter(Sport %in% tabla_medallas$Sport)

tabla_edad <- df_top_sports %>%
  group_by(Sport) %>%
  summarise(
    Edad_minima = min(Age, na.rm = TRUE),
    Edad_promedio = round(mean(Age, na.rm = TRUE), 1),
    Edad_mediana = median(Age, na.rm = TRUE),
    Edad_maxima = max(Age, na.rm = TRUE)
  )

tabla_edad

,Sport,Edad_minima,Edad_promedio,Edad_mediana,Edad_maxima
1,Athletics,19,26.5,26,40
2,Football,18,24.3,24,38
3,Hockey,19,26.8,27,37
4,Rowing,20,28.1,28,38
5,Swimming,16,23.3,23,35


### 2.3 Equipos nacionales con más medallas
Top 10 de delegaciones y países con mayor cantidad de preseas obtenidas en los deportes analizados.

In [5]:
tabla_equipos <- df_top_sports %>%
  count(Team, sort = TRUE) %>%
  head(10)

tabla_equipos

,Team,n
1,United States,119
2,Germany,87
3,Great Britain,75
4,Canada,41
5,Australia,39
6,Netherlands,36
7,Brazil,35
8,Jamaica,30
9,New Zealand,23
10,China,20


### 2.4 Análisis del peso por sexo
Comparación estadística y visual del peso corporal de los atletas galardonados según el género mediante boxplot.

In [6]:
df_peso <- df_top_sports %>%
  filter(!is.na(Weight))

tabla_peso <- df_peso %>%
  group_by(Sex) %>%
  summarise(
    Peso_minimo = min(Weight, na.rm = TRUE),
    Peso_promedio = round(mean(Weight, na.rm = TRUE), 1),
    Peso_mediano = median(Weight, na.rm = TRUE),
    Peso_maximo = max(Weight, na.rm = TRUE)
  )

tabla_peso

df_peso %>%
  ggplot(mapping = aes(x = Sex, y = Weight, fill = Sex)) +
  geom_boxplot(alpha = 0.7, show.legend = FALSE) +
  labs(
    title = "Distribución del peso de los medallistas en 2016",
    x = "Sexo",
    y = "Peso (kg)"
  ) +
  theme_bw()

,Sex,Peso_minimo,Peso_promedio,Peso_mediano,Peso_maximo
1,F,43,64.2,63,136
2,M,52,79.8,78,145


[Boxplot: Distribución del peso de los medallistas por Sexo (F y M)]

---
## Ejercicio 3 {#ejercicio-3}

Transformación y estructuración tabular del archivo `us_state_population.tsv` utilizando las funciones `pivot_longer`, `unite` y `separate` de `tidyr`.

In [7]:
df_pob <- read_tsv("us_state_population.tsv", show_col_types = FALSE)

df_nuevo <- df_pob %>%
  pivot_longer(
    cols = `2010`:`2018`,
    names_to = "Year",
    values_to = "Population"
  ) %>%
  unite("State_Code", State, Code, sep = "_") %>%
  separate(State_Code, into = c("State", "Code"), sep = "_")

head(df_nuevo, 15)

,State,Code,Year,Population
1,Alabama,AL,2010,4785401
2,Alabama,AL,2011,4799918
3,Alabama,AL,2012,4815960
4,Alabama,AL,2013,4830382
5,Alabama,AL,2014,4846411
6,Alabama,AL,2015,4858979
7,Alabama,AL,2016,4863300
8,Alabama,AL,2017,4874747
9,Alabama,AL,2018,4887871
10,Alaska,AK,2010,713910


---
## Ejercicio 4 {#ejercicio-4}

Generación de un histograma para analizar la distribución y el sesgo del total de acres quemados en eventos de gran magnitud ($\ge 1.000$ acres).

In [8]:
datos_incendios <- read_csv("StudyArea.csv", show_col_types = FALSE)

datos_incendios %>%
  select(ORGANIZATI, STATE, YEAR_, TOTALACRES, CAUSE) %>%
  filter(TOTALACRES >= 1000) %>%
  ggplot(mapping = aes(x = TOTALACRES)) +
  geom_histogram(
    binwidth = 500,
    fill = "steelblue",
    color = "white"
  ) +
  coord_cartesian(xlim = c(1000, 30000)) +
  labs(
    title = "Distribución de acres quemados",
    subtitle = "Incendios forestales de 1.000 acres o más",
    x = "Total de acres quemados",
    y = "Número de incendios"
  ) +
  theme_bw()

[Histograma ggplot2: Distribución de acres quemados >= 1000 acres]

---
## Conclusiones

- **Ejercicio 1:** Permitió analizar series temporales interactivas con `plotly`, evidenciando el impacto acumulado y patrones históricos por tipo de causa en Idaho.
- **Ejercicio 2:** Demostró el flujo de agregación con `dplyr` para evaluar perfiles demográficos (edad y peso) de atletas olímpicos.
- **Ejercicio 3:** Mostró la reestructuración de formato ancho a largo (*tidy data*) mediante `pivot_longer` y manipulación de cadenas con `unite`/`separate`.
- **Ejercicio 4:** Evidenció el sesgo a la derecha en la distribución de magnitud de incendios forestales mediante técnicas de ajuste de rango en `ggplot2`.

*Fin de los ejercicios del Módulo 2.*
